<a href="https://colab.research.google.com/github/ashwiniv07/A-Synergistic-Deep-Fusion-and-Adversarial-Defense-Framework-/blob/main/protocol_B_paysim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install lightgbm xgboost imbalanced-learn statsmodels

In [ ]:
!ls "/content/drive/MyDrive"
!find /content/drive/MyDrive -maxdepth 3 -name "protocol_b.py"
!find /content/drive/MyDrive -maxdepth 3 -type d -name "protocol_b_results"

 Ashwini.pdf
 Ashwini.pdf.pdf
'Colab Notebooks'
'Copy of ieee_cis_fraud_dataset(23_10)new.ipynb'
'ieee_cis_fraud_dataset(23_10)new.ipynb'
 SCORE_FALL26_27-REVIEW-1.xlsx
'second paper'
/content/drive/MyDrive/second paper/protocol_b.py


In [ ]:
!ls -la "/content/drive/MyDrive/second paper"
!ls "/content/drive/MyDrive/second paper/data"
!ls "/content/drive/MyDrive/second paper/protocol_b_results/ieee" | head

total 67900
drwx------ 2 root root     4096 Nov 13 08:37  .
drwx------ 4 root root     4096 Nov 13 08:28  ..
-rw------- 1 root root       67 Nov 11 08:50  kaggle.json
-rw------- 1 root root     3300 Nov 11 09:09  ProtocolB_PaySim_2.ipynb
-rw------- 1 root root    41793 Nov 13 08:35  protocol_b.py
-rw------- 1 root root    41561 Nov 11 08:28 'protocol B.py'
-rw------- 1 root root 69432528 Nov 13 08:34  protocol_b_results-20261113T083408Z-1-001.zip
ls: cannot access '/content/drive/MyDrive/second paper/data': No such file or directory
ls: cannot access '/content/drive/MyDrive/second paper/protocol_b_results/ieee': No such file or directory


In [ ]:
import os, shutil, glob
BASE = '/content/drive/MyDrive/second paper'
DATA = BASE + '/data'
os.makedirs(DATA, exist_ok=True)

# 1) unpack the saved results zip
!unzip -o -q "{BASE}/protocol_b_results-20261113T083408Z-1-001.zip" -d "{BASE}"

# 2) download PaySim from Kaggle
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy(BASE + '/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
!kaggle datasets download -d ealaxi/paysim1 -p /content/paysim --unzip
shutil.copy(glob.glob('/content/paysim/*.csv')[0], DATA + '/paysimdataset.csv')

!ls "{BASE}"
!ls -lh "{DATA}"

Dataset URL: https://www.kaggle.com/datasets/ealaxi/paysim1
License(s): CC-BY-SA-4.0
100% 178M/178M [00:00<00:00, 191MB/s]

 data			   'protocol B.py'
 kaggle.json		    protocol_b_results
 ProtocolB_PaySim_2.ipynb   protocol_b_results-20261113T083408Z-1-001.zip
 protocol_b.py
total 471M
-rw------- 1 root root 471M Nov 13 08:41 paysimdataset.csv


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, importlib
BASE = '/content/drive/MyDrive/second paper'
DATA_DIR = BASE + '/data'
sys.path.insert(0, BASE)
import protocol_b as pb
importlib.reload(pb)
pb.init(BASE + '/protocol_b_results/paysim', dataset='paysim', smoke=False,
        seeds_main=[0, 1, 2], pool_legit_cap=50000, max_epochs=100)
import tensorflow as tf
print('GPU:', tf.config.list_physical_devices('GPU'))

Results -> /content/drive/MyDrive/second paper/protocol_b_results/paysim | dataset paysim 
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
S = pb.chrono_split(pb.load_paysim(DATA_DIR + '/paysimdataset.csv'))
pb.register(S)

PaySim: (6362620, 7) fraud rate 0.00129
  tr: n=4453834  fraud=3643 (0.0008)
  va: n=636262  fraud=316 (0.0005)
  te: n=1272524  fraud=4254 (0.0033)


In [ ]:
pb.run_all(S, paysim=True, tables=(7,))


=== Table 7: main comparison ===
  xgb                    seed 0  PR-AUC 0.9678  R@1%FPR 0.9976
  lgbm                   seed 0  PR-AUC 0.9172  R@1%FPR 0.9779
  NeuroSync: kept 3643 legit + 3643 fraud
  training cnn / neurosync / seed 0 on 7286 rows (3643 fraud)
    epoch 1  val PR-AUC 0.0011  (best 0.0011)  12s
    epoch 2  val PR-AUC 0.0012  (best 0.0012)  0s
    epoch 3  val PR-AUC 0.0014  (best 0.0014)  0s
    epoch 4  val PR-AUC 0.0022  (best 0.0022)  0s
    epoch 5  val PR-AUC 0.0041  (best 0.0041)  0s
    epoch 6  val PR-AUC 0.0071  (best 0.0071)  0s
    epoch 7  val PR-AUC 0.0129  (best 0.0129)  0s
    epoch 8  val PR-AUC 0.0505  (best 0.0505)  0s
    epoch 9  val PR-AUC 0.0894  (best 0.0894)  0s
    epoch 10  val PR-AUC 0.1191  (best 0.1191)  0s
    epoch 11  val PR-AUC 0.1343  (best 0.1343)  0s
    epoch 12  val PR-AUC 0.1911  (best 0.1911)  0s
    epoch 13  val PR-AUC 0.1897  (best 0.1911)  0s
    epoch 14  val PR-AUC 0.1600  (best 0.1911)  0s
    epoch 15  val PR-AUC 0.236

In [ ]:
pb.run_all(S, paysim=True, tables=(8,))


=== Table 8: imbalance treatments ===
  training proposed / classweight / seed 0 on 53643 rows (3643 fraud)
    epoch 1  val PR-AUC 0.1434  (best 0.1434)  16s
    epoch 2  val PR-AUC 0.3898  (best 0.3898)  6s
    epoch 3  val PR-AUC 0.5040  (best 0.5040)  7s
    epoch 4  val PR-AUC 0.6036  (best 0.6036)  6s
    epoch 5  val PR-AUC 0.6244  (best 0.6244)  6s
    epoch 6  val PR-AUC 0.6422  (best 0.6422)  5s
    epoch 7  val PR-AUC 0.6125  (best 0.6422)  5s
    epoch 8  val PR-AUC 0.6474  (best 0.6474)  15s
    epoch 9  val PR-AUC 0.6337  (best 0.6474)  6s
    epoch 10  val PR-AUC 0.6496  (best 0.6496)  6s
    epoch 11  val PR-AUC 0.6379  (best 0.6496)  5s
    epoch 12  val PR-AUC 0.6456  (best 0.6496)  5s
    epoch 13  val PR-AUC 0.6376  (best 0.6496)  5s
    epoch 14  val PR-AUC 0.6422  (best 0.6496)  15s
    epoch 15  val PR-AUC 0.6141  (best 0.6496)  6s
  -> proposed seed 0  PR-AUC 0.0196  R@1%FPR 0.0000  (15 epochs)
  training proposed / under / seed 0 on 7286 rows (3643 fraud)
    

In [ ]:
pb.run_all(S, paysim=True, tables=(8,))


=== Table 8: imbalance treatments ===
  training proposed / neurosync_density / seed 1 on 7286 rows (3643 fraud)
    epoch 1  val PR-AUC 0.0009  (best 0.0009)  20s
    epoch 2  val PR-AUC 0.0010  (best 0.0010)  1s
    epoch 3  val PR-AUC 0.0009  (best 0.0010)  1s
    epoch 4  val PR-AUC 0.0009  (best 0.0010)  1s
    epoch 5  val PR-AUC 0.0009  (best 0.0010)  1s
    epoch 6  val PR-AUC 0.0010  (best 0.0010)  1s
    epoch 7  val PR-AUC 0.0010  (best 0.0010)  1s
    epoch 8  val PR-AUC 0.0010  (best 0.0010)  1s
    epoch 9  val PR-AUC 0.0011  (best 0.0011)  1s
    epoch 10  val PR-AUC 0.0011  (best 0.0011)  2s
    epoch 11  val PR-AUC 0.0012  (best 0.0012)  1s
    epoch 12  val PR-AUC 0.0012  (best 0.0012)  1s
    epoch 13  val PR-AUC 0.0012  (best 0.0012)  1s
    epoch 14  val PR-AUC 0.0013  (best 0.0013)  1s
    epoch 15  val PR-AUC 0.0014  (best 0.0014)  1s
    epoch 16  val PR-AUC 0.0014  (best 0.0014)  1s
    epoch 17  val PR-AUC 0.0014  (best 0.0014)  1s
    epoch 18  val PR-AUC 0.

In [ ]:
pb.run_all(S, paysim=True, tables=(9,))


=== Table 9: ablation ===
  training no_cnn / neurosync / seed 0 on 7286 rows (3643 fraud)
    epoch 1  val PR-AUC 0.0010  (best 0.0010)  12s
    epoch 2  val PR-AUC 0.0015  (best 0.0015)  1s
    epoch 3  val PR-AUC 0.0019  (best 0.0019)  1s
    epoch 4  val PR-AUC 0.0033  (best 0.0033)  1s
    epoch 5  val PR-AUC 0.0050  (best 0.0050)  1s
    epoch 6  val PR-AUC 0.0163  (best 0.0163)  1s
    epoch 7  val PR-AUC 0.0304  (best 0.0304)  1s
    epoch 8  val PR-AUC 0.0837  (best 0.0837)  1s
    epoch 9  val PR-AUC 0.1059  (best 0.1059)  1s
    epoch 10  val PR-AUC 0.1057  (best 0.1059)  2s
    epoch 11  val PR-AUC 0.1223  (best 0.1223)  1s
    epoch 12  val PR-AUC 0.1311  (best 0.1311)  1s
    epoch 13  val PR-AUC 0.1538  (best 0.1538)  1s
    epoch 14  val PR-AUC 0.1754  (best 0.1754)  1s
    epoch 15  val PR-AUC 0.2135  (best 0.2135)  1s
    epoch 16  val PR-AUC 0.2967  (best 0.2967)  1s
    epoch 17  val PR-AUC 0.2031  (best 0.2967)  1s
    epoch 18  val PR-AUC 0.1617  (best 0.2967)  1

In [ ]:
pb.run_all(S, paysim=True, tables=(10,))


=== Table 10: attacks ===
  attacks cnn seed 0: {'clean': 0.6465, 'fgsm_0.01': 0.6063, 'fgsm_0.05': 0.3907, 'fgsm_0.1': 0.2064, 'pgd_0.05': 0.3907, 'cpgd_0.1': 0.6331}
  attacks no_adv seed 0: {'clean': 0.7085, 'fgsm_0.01': 0.6916, 'fgsm_0.05': 0.5654, 'fgsm_0.1': 0.2518, 'pgd_0.05': 0.5635, 'cpgd_0.1': 0.6902}
  attacks proposed seed 0: {'clean': 0.7487, 'fgsm_0.01': 0.7304, 'fgsm_0.05': 0.6335, 'fgsm_0.1': 0.5038, 'pgd_0.05': 0.6335, 'cpgd_0.1': 0.7313}
  attacks cnn seed 1: {'clean': 0.6399, 'fgsm_0.01': 0.6321, 'fgsm_0.05': 0.5874, 'fgsm_0.1': 0.5101, 'pgd_0.05': 0.5846, 'cpgd_0.1': 0.6185}
  attacks no_adv seed 1: {'clean': 0.8091, 'fgsm_0.01': 0.7889, 'fgsm_0.05': 0.7102, 'fgsm_0.1': 0.6368, 'pgd_0.05': 0.7073, 'cpgd_0.1': 0.7713}
  attacks proposed seed 1: {'clean': 0.7854, 'fgsm_0.01': 0.7666, 'fgsm_0.05': 0.6747, 'fgsm_0.1': 0.5839, 'pgd_0.05': 0.6737, 'cpgd_0.1': 0.7513}
  attacks cnn seed 2: {'clean': 0.7212, 'fgsm_0.01': 0.6982, 'fgsm_0.05': 0.6117, 'fgsm_0.1': 0.5155, 'pg

In [ ]:
T = pb.summary(paysim=True)
for name, df in T.items():
    print('\n' + name); display(df)

Saved /content/drive/MyDrive/second paper/protocol_b_results/paysim/tables_paysim.xlsx

t7


,Model,PR-AUC,ROC-AUC,Recall@1% FPR,Recall@0.1% FPR,Precision,Recall,F1,MCC
0,Reference rule,0.0033,0.3100,0.0000,0.0000,0.0000,0.0031,0.0001,-0.0451
1,Logistic regression,0.6299 ± 0.0000,0.9664 ± 0.0000,0.7388 ± 0.0000,0.5658 ± 0.0000,0.9452 ± 0.0000,0.4382 ± 0.0000,0.5988 ± 0.0000,0.6429 ± 0.0000
2,XGBoost,0.9681 ± 0.0010,0.9998 ± 0.0000,0.9977 ± 0.0001,0.9692 ± 0.0018,0.9922 ± 0.0016,0.7443 ± 0.0020,0.8506 ± 0.0007,0.8590 ± 0.0005
3,LightGBM,0.5684 ± 0.3579,0.8637 ± 0.1786,0.5330 ± 0.4949,0.4992 ± 0.4552,0.7211 ± 0.4317,0.6903 ± 0.2158,0.6001 ± 0.2256,0.6471 ± 0.1862
4,CNN,0.5199 ± 0.0773,0.9111 ± 0.0741,0.6102 ± 0.0334,0.5001 ± 0.1038,0.1106 ± 0.0636,0.6692 ± 0.0452,0.1833 ± 0.0977,0.2537 ± 0.0863
5,BiGRU,0.1338 ± 0.2063,0.8475 ± 0.0103,0.1301 ± 0.2253,0.1228 ± 0.2127,0.0125 ± 0.0051,0.7908 ± 0.1479,0.0245 ± 0.0097,0.0787 ± 0.0126
6,Transformer,0.3749 ± 0.0922,0.8971 ± 0.0289,0.5446 ± 0.0740,0.4126 ± 0.0631,0.7822 ± 0.0299,0.3433 ± 0.0511,0.4763 ± 0.0556,0.5167 ± 0.0489
7,Proposed (full),0.3345 ± 0.1966,0.8987 ± 0.0433,0.4410 ± 0.1697,0.2533 ± 0.2796,0.0249 ± 0.0088,0.7526 ± 0.0310,0.0481 ± 0.0163,0.1232 ± 0.0229



t8


,Imbalance treatment,PR-AUC,Recall@1% FPR,Sampling time (s)
0,Class-weighted loss,0.3733 ± 0.3233,0.3894 ± 0.3632,0 ± 0
1,Random undersampling,0.1266 ± 0.1746,0.1731 ± 0.2998,0 ± 0
2,SMOTE,0.5397 ± 0.0695,0.5570 ± 0.0713,1 ± 1
3,"NeuroSync (full fitness, Eq. 5)",0.3345 ± 0.1966,0.4410 ± 0.1697,2 ± 1
4,"NeuroSync (density only, γ = δ = 0)",0.0208 ± 0.0079,0.0300 ± 0.0520,2 ± 0



t9


,Variant,PR-AUC
0,Full model,0.3345 ± 0.1966
1,Without CNN branch,0.1786 ± 0.2706
2,Without BiGRU branch,0.5247 ± 0.1245
3,Without Transformer branch,0.2272 ± 0.1363
4,Attention fusion replaced by concatenation,0.5100 ± 0.1539
5,Attention fusion replaced by averaging of bran...,0.5358 ± 0.1624
6,Ensemble of the three separately trained branc...,0.6348 ± 0.0265
7,"Without adversarial training (γ = 0, no discri...",0.3076 ± 0.2103
8,FGSM training without the GRL discriminator,0.2368 ± 0.1925
9,"Full model, columns randomly permuted",0.3564 ± 0.0871


... (2 outputs left out)


t11


,Model,Trainable parameters,Training time per epoch (s),"Inference latency, batch 1 (ms)","Throughput, batch 1,024 (transactions/s)"
0,LightGBM,–,–,1.19,694372
1,CNN,12642,0.3,1.45,603187
2,BiGRU,36130,0.4,1.64,363160
3,Transformer,48226,0.7,1.38,285544
4,Proposed,138274,1.7,3.68,131654



disc


,disc_accuracy,disc_auc,disc_bce,mean_clean,mean_adv
0,0.5050 ± 0.0149,0.5004 ± 0.0113,0.6985 ± 0.0034,0.4905 ± 0.0095,0.4901 ± 0.0110


In [ ]:
import os, sys, importlib, warnings; warnings.filterwarnings('ignore')
BASE = '/content/drive/MyDrive/second paper'
f = BASE + '/protocol_b.py'
s = open(f).read()
a1 = "newbalanceOrig=df['newbalanceOrig'].values)"
b1 = "newbalanceOrig=df['newbalanceOrig'].values, oldbalanceDest=df['oldbalanceDest'].values, newbalanceDest=df['newbalanceDest'].values)"
a2 = "bad = np.abs(raw['oldbalanceOrg'][te] - raw['amount'][te] - raw['newbalanceOrig'][te]) > 0.01"
b2 = "bad = np.abs(raw['oldbalanceDest'][te] + raw['amount'][te] - raw['newbalanceDest'][te]) > 0.01"
print('fix 1 found:', a1 in s, '| fix 2 found:', a2 in s)
s = s.replace(a1, b1).replace(a2, b2); open(f, 'w').write(s)
k = BASE + '/protocol_b_results/paysim/runs/paysim__rule__none__seed0.json'
if os.path.exists(k): os.remove(k)

sys.path.insert(0, BASE); import protocol_b as pb; importlib.reload(pb)
pb.init(BASE + '/protocol_b_results/paysim', dataset='paysim', smoke=False,
        seeds_main=[0, 1, 2], pool_legit_cap=50000, max_epochs=100)
S = pb.chrono_split(pb.load_paysim(BASE + '/data/paysimdataset.csv')); pb.register(S)
print(pb.run_reference_rule(S))
T = pb.summary(paysim=True); display(T['t7'])

fix 1 found: True | fix 2 found: True
Results -> /content/drive/MyDrive/second paper/protocol_b_results/paysim | dataset paysim 
PaySim: (6362620, 7) fraud rate 0.00129
  tr: n=4453834  fraud=3643 (0.0008)
  va: n=636262  fraud=316 (0.0005)
  te: n=1272524  fraud=4254 (0.0033)
{'pr_auc': 0.01625635028170271, 'roc_auc': 0.7419210752639886, 'rec_fpr1': 0.0, 'rec_fpr01': 0.0, 'precision': 0.02677877804688571, 'recall': 0.5510108133521392, 'f1': 0.05107532739197699, 'mcc': 0.11034868632821088, 'threshold': 0.5}
Saved /content/drive/MyDrive/second paper/protocol_b_results/paysim/tables_paysim.xlsx


,Model,PR-AUC,ROC-AUC,Recall@1% FPR,Recall@0.1% FPR,Precision,Recall,F1,MCC
0,Reference rule,0.0163,0.7419,0.0000,0.0000,0.0268,0.5510,0.0511,0.1103
1,Logistic regression,0.6299 ± 0.0000,0.9664 ± 0.0000,0.7388 ± 0.0000,0.5658 ± 0.0000,0.9452 ± 0.0000,0.4382 ± 0.0000,0.5988 ± 0.0000,0.6429 ± 0.0000
2,XGBoost,0.9681 ± 0.0010,0.9998 ± 0.0000,0.9977 ± 0.0001,0.9692 ± 0.0018,0.9922 ± 0.0016,0.7443 ± 0.0020,0.8506 ± 0.0007,0.8590 ± 0.0005
3,LightGBM,0.5684 ± 0.3579,0.8637 ± 0.1786,0.5330 ± 0.4949,0.4992 ± 0.4552,0.7211 ± 0.4317,0.6903 ± 0.2158,0.6001 ± 0.2256,0.6471 ± 0.1862
4,CNN,0.5199 ± 0.0773,0.9111 ± 0.0741,0.6102 ± 0.0334,0.5001 ± 0.1038,0.1106 ± 0.0636,0.6692 ± 0.0452,0.1833 ± 0.0977,0.2537 ± 0.0863
5,BiGRU,0.1338 ± 0.2063,0.8475 ± 0.0103,0.1301 ± 0.2253,0.1228 ± 0.2127,0.0125 ± 0.0051,0.7908 ± 0.1479,0.0245 ± 0.0097,0.0787 ± 0.0126
6,Transformer,0.3749 ± 0.0922,0.8971 ± 0.0289,0.5446 ± 0.0740,0.4126 ± 0.0631,0.7822 ± 0.0299,0.3433 ± 0.0511,0.4763 ± 0.0556,0.5167 ± 0.0489
7,Proposed (full),0.3345 ± 0.1966,0.8987 ± 0.0433,0.4410 ± 0.1697,0.2533 ± 0.2796,0.0249 ± 0.0088,0.7526 ± 0.0310,0.0481 ± 0.0163,0.1232 ± 0.0229
